# Ambiente conda da aula

Rode no Anaconda Prompt ou PowerShell, a partir da raiz da pasta `4_Ciclo_Vida_MLOps_Bitcoin`, antes de executar este notebook.

Todos os arquivos gerados pela execução ficam em `runtime/`. Para limpar o projeto, apague essa pasta.

Crie um ambiente conda novo com Python 3.12.

```powershell
conda create -n aula_4_mlops_bitcoin python=3.12 --no-default-packages -y

conda activate aula_4_mlops_bitcoin

conda install pip -y

python -m pip install --upgrade pip

pip install -r 02_api_bitcoin/requirements.txt

pip install matplotlib==3.10.9 seaborn==0.13.2 scipy==1.17.1 ipykernel==7.2.0 jupyter==1.1.1

python -m ipykernel install --user --name aula_4_mlops_bitcoin --display-name "Python (aula_4_mlops_bitcoin)"
```


Depois selecione o kernel `Python (aula_4_mlops_bitcoin)` no Jupyter.

# 01 — Treinamento, pipeline e MLflow

**Objetivo:** treinar uma pipeline para prever o `PriceUSD` do Bitcoin no próximo dia.

Este notebook mostra o fluxo de MLOps:

- baixar dados;
- criar features sem vazamento temporal;
- separar treino, validação e teste por tempo;
- comparar baseline e modelos candidatos;
- registrar métricas, artefatos e modelo no MLflow;
- publicar o modelo no catálogo com alias `homol`.

In [1]:
from pathlib import Path
import json
import os
from datetime import datetime
import warnings

import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

In [2]:
def find_lesson_dir():
    current = Path.cwd()
    if current.name == "4_Ciclo_Vida_MLOps_Bitcoin":
        return current
    candidate = current / "4_Ciclo_Vida_MLOps_Bitcoin"
    if candidate.exists():
        return candidate
    return current


LESSON_DIR = find_lesson_dir()
RUNTIME_DIR = LESSON_DIR / "runtime"
DATA_DIR = RUNTIME_DIR / "data"
ARTIFACTS_DIR = RUNTIME_DIR / "artifacts"
MLFLOW_RUNS_DIR = RUNTIME_DIR / "mlruns"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_RUNS_DIR.mkdir(parents=True, exist_ok=True)

BTC_URL = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv"
MODEL_NAME = "bitcoin_price_forecaster"
EXPERIMENT_NAME = "aula_4_bitcoin_price_forecasting"
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", f"sqlite:///{RUNTIME_DIR / 'mlflow.db'}")

print("Lesson dir:", LESSON_DIR)
print("MLflow tracking URI:", TRACKING_URI)

Lesson dir: c:\Users\apcs1\Downloads\Operações de Aprendizado de Machine Learning\4_Ciclo_Vida_MLOps_Bitcoin - Copia
MLflow tracking URI: sqlite:///c:\Users\apcs1\Downloads\Operações de Aprendizado de Machine Learning\4_Ciclo_Vida_MLOps_Bitcoin - Copia\runtime\mlflow.db


## Carregar dados

A base vem do projeto público CoinMetrics.

O período de treino inicial vai de `2018-01-01` até `2025-12-31`.

In [3]:
def load_btc_data(start_date, end_date):
    raw_path = DATA_DIR / "btc.csv"
    if raw_path.exists():
        data = pd.read_csv(raw_path)
    else:
        data = pd.read_csv(BTC_URL)
        data.to_csv(raw_path, index=False)

    data["time"] = pd.to_datetime(data["time"])
    data = data.sort_values("time").reset_index(drop=True)
    mask = (data["time"] >= pd.Timestamp(start_date)) & (data["time"] <= pd.Timestamp(end_date))
    return data.loc[mask].copy()


btc = load_btc_data("2018-01-01", "2025-12-31")
print(btc.shape)
btc[["time", "PriceUSD"]].head()

(2922, 32)


,time,PriceUSD
3285,2018-01-01,13464.653612
3286,2018-01-02,14754.322205
3287,2018-01-03,15010.286160
3288,2018-01-04,15070.300799
3289,2018-01-05,16997.227408


## Criar features sem vazamento

O alvo é o preço do próximo dia:

```python
target_price_usd_d1 = PriceUSD.shift(-1)
```

As features usam apenas informações disponíveis até o dia atual.

Nesta aula, isso significa que o preço do dia atual já é conhecido no momento da inferência. A previsão é feita depois do fechamento ou depois da consolidação diária da base.

Se a previsão precisasse acontecer antes do fechamento do dia, `price_usd_current`, retornos e janelas móveis teriam que ser deslocados com `shift(1)`.


In [4]:
EXTRA_NUMERIC_COLUMNS = [
    "AdrActCnt",
    "AdrBalCnt",
    "BlkCnt",
    "CapMrktCurUSD",
    "HashRate",
    "ROI30d",
    "ROI1yr",
    "SplyCur",
    "TxCnt",
    "TxTfrCnt",
    "volume_reported_spot_usd_1d",
]


def build_feature_table(data):
    frame = data.copy()
    frame["target_price_usd_d1"] = frame["PriceUSD"].shift(-1)
    frame["price_usd_current"] = frame["PriceUSD"]

    for lag in [1, 2, 3, 7, 14, 30]:
        frame[f"price_usd_lag_{lag}"] = frame["PriceUSD"].shift(lag)

    frame["return_1d"] = frame["PriceUSD"].pct_change()
    frame["log_return_1d"] = np.log(frame["PriceUSD"]).diff()

    for window in [7, 14, 30]:
        frame[f"rolling_mean_{window}"] = frame["PriceUSD"].rolling(window).mean()
        frame[f"rolling_std_{window}"] = frame["PriceUSD"].rolling(window).std()
        frame[f"rolling_min_{window}"] = frame["PriceUSD"].rolling(window).min()
        frame[f"rolling_max_{window}"] = frame["PriceUSD"].rolling(window).max()
        frame[f"rolling_return_{window}"] = frame["PriceUSD"].pct_change(window)

    frame["day_of_week"] = frame["time"].dt.dayofweek
    frame["month"] = frame["time"].dt.month
    frame["quarter"] = frame["time"].dt.quarter
    frame["year"] = frame["time"].dt.year

    feature_columns = [
        "price_usd_current",
        "price_usd_lag_1",
        "price_usd_lag_2",
        "price_usd_lag_3",
        "price_usd_lag_7",
        "price_usd_lag_14",
        "price_usd_lag_30",
        "return_1d",
        "log_return_1d",
        "day_of_week",
        "month",
        "quarter",
        "year",
    ]

    for window in [7, 14, 30]:
        feature_columns.extend(
            [
                f"rolling_mean_{window}",
                f"rolling_std_{window}",
                f"rolling_min_{window}",
                f"rolling_max_{window}",
                f"rolling_return_{window}",
            ]
        )

    for column in EXTRA_NUMERIC_COLUMNS:
        if column in frame.columns and frame[column].notna().mean() >= 0.80:
            feature_columns.append(column)

    selected = frame[["time", "target_price_usd_d1"] + feature_columns].copy()
    selected = selected.dropna(subset=["target_price_usd_d1"] + feature_columns).reset_index(drop=True)
    return selected, feature_columns


supervised, feature_columns = build_feature_table(btc)
print(supervised.shape)
print(feature_columns)
supervised.head()

(2891, 41)
['price_usd_current', 'price_usd_lag_1', 'price_usd_lag_2', 'price_usd_lag_3', 'price_usd_lag_7', 'price_usd_lag_14', 'price_usd_lag_30', 'return_1d', 'log_return_1d', 'day_of_week', 'month', 'quarter', 'year', 'rolling_mean_7', 'rolling_std_7', 'rolling_min_7', 'rolling_max_7', 'rolling_return_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_min_14', 'rolling_max_14', 'rolling_return_14', 'rolling_mean_30', 'rolling_std_30', 'rolling_min_30', 'rolling_max_30', 'rolling_return_30', 'AdrActCnt', 'AdrBalCnt', 'BlkCnt', 'CapMrktCurUSD', 'HashRate', 'ROI30d', 'ROI1yr', 'SplyCur', 'TxCnt', 'TxTfrCnt', 'volume_reported_spot_usd_1d']


,time,target_price_usd_d1,price_usd_current,price_usd_lag_1,price_usd_lag_2,price_usd_lag_3,price_usd_lag_7,price_usd_lag_14,price_usd_lag_30,return_1d,log_return_1d,day_of_week,month,quarter,year,rolling_mean_7,rolling_std_7,rolling_min_7,rolling_max_7,rolling_return_7,rolling_mean_14,rolling_std_14,rolling_min_14,rolling_max_14,rolling_return_14,rolling_mean_30,rolling_std_30,rolling_min_30,rolling_max_30,rolling_return_30,AdrActCnt,AdrBalCnt,BlkCnt,CapMrktCurUSD,HashRate,ROI30d,ROI1yr,SplyCur,TxCnt,TxTfrCnt,volume_reported_spot_usd_1d
0,2018-01-31,9039.865075,10050.095797,9959.546715,11132.554314,11615.958711,11225.896175,11159.608591,13464.653612,0.009092,0.009051,2,1,1,2018,10892.163611,635.313977,9959.546715,11615.958711,-0.104740,11138.205575,675.873325,9959.546715,12768.404586,-0.099422,12871.940750,2083.739419,9959.546715,17103.58928,-0.253594,685452.0,27206731.0,117.0,1.692194e+11,1.513976e+07,-25.359418,937.697995,1.683759e+07,208360.0,552597.0,3.356067e+09
1,2018-02-01,8786.077380,9039.865075,10050.095797,9959.546715,11132.554314,11130.464947,11246.039730,14754.322205,-0.100520,-0.105938,3,2,1,2018,10593.506487,928.403010,9039.865075,11615.958711,-0.187827,10980.621671,876.277477,9039.865075,12768.404586,-0.196173,12681.458846,2165.322883,9039.865075,17103.58928,-0.387307,833934.0,27208439.0,160.0,1.522276e+11,2.070395e+07,-38.730733,815.399164,1.683959e+07,252498.0,660268.0,5.030838e+09
2,2018-02-02,9132.100322,8786.077380,9039.865075,10050.095797,9959.546715,11035.554610,11440.255562,15010.286160,-0.028074,-0.028476,4,2,1,2018,10272.152597,1119.531752,8786.077380,11615.958711,-0.203839,10791.037515,1040.849506,8786.077380,12768.404586,-0.232003,12473.985220,2231.663006,8786.077380,17103.58928,-0.414663,853582.0,27142662.0,168.0,1.479724e+11,2.173915e+07,-41.466290,771.890375,1.684169e+07,237739.0,632968.0,6.299179e+09
3,2018-02-03,8250.526896,9132.100322,8786.077380,9039.865075,10050.095797,11320.970185,12768.404586,15070.300799,0.039383,0.038627,5,2,1,2018,9959.456902,1082.848000,8786.077380,11615.958711,-0.193346,10531.301496,960.023754,8786.077380,11615.958711,-0.284789,12276.045204,2256.646932,8786.077380,17103.58928,-0.394033,714800.0,27062866.0,158.0,1.538180e+11,2.044515e+07,-39.403331,799.446664,1.684366e+07,190833.0,498555.0,3.033162e+09
4,2018-02-04,6849.542559,8250.526896,9132.100322,8786.077380,9039.865075,11615.958711,11422.394259,16997.227408,-0.096536,-0.101519,6,2,1,2018,9478.680929,965.555411,8250.526896,11132.554314,-0.289725,10304.739542,1097.923035,8250.526896,11615.958711,-0.277688,11984.488520,2189.680209,8250.526896,17103.58928,-0.514596,711645.0,26935765.0,178.0,1.389874e+11,2.303314e+07,-51.459572,698.088493,1.684589e+07,172621.0,454924.0,3.260391e+09


## Separar treino, validação e teste

Como é série temporal, a separação é cronológica.

Não usamos `shuffle`.

In [5]:
train_data = supervised[supervised["time"] <= "2023-12-31"].copy()
valid_data = supervised[(supervised["time"] >= "2024-01-01") & (supervised["time"] <= "2024-12-31")].copy()
test_data = supervised[(supervised["time"] >= "2025-01-01") & (supervised["time"] <= "2025-12-31")].copy()

X_train = train_data[feature_columns]
y_train = train_data["target_price_usd_d1"]
X_valid = valid_data[feature_columns]
y_valid = valid_data["target_price_usd_d1"]
X_test = test_data[feature_columns]
y_test = test_data["target_price_usd_d1"]

print("Train:", X_train.shape, train_data["time"].min(), train_data["time"].max())
print("Valid:", X_valid.shape, valid_data["time"].min(), valid_data["time"].max())
print("Test :", X_test.shape, test_data["time"].min(), test_data["time"].max())

Train: (2161, 39) 2018-01-31 00:00:00 2023-12-31 00:00:00
Valid: (366, 39) 2024-01-01 00:00:00 2024-12-31 00:00:00
Test : (364, 39) 2025-01-01 00:00:00 2025-12-30 00:00:00


## Métricas

O baseline prevê o preço de amanhã como o preço de hoje.

O modelo precisa superar esse baseline para justificar sua complexidade.

In [6]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1, denominator)
    return float(np.mean(np.abs(y_true - y_pred) / denominator))


def calculate_metrics(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "smape": smape(y_true, y_pred),
        "r2": float(r2_score(y_true, y_pred)),
    }


baseline_valid = X_valid["price_usd_current"].to_numpy()
baseline_test = X_test["price_usd_current"].to_numpy()

results = [
    {
        "model": "naive_current_price",
        "valid": calculate_metrics(y_valid, baseline_valid),
        "test": calculate_metrics(y_test, baseline_test),
    }
]

results[0]

{'model': 'naive_current_price',
 'valid': {'mae': 1316.7434360270856,
  'rmse': 1834.7949489867306,
  'smape': 0.019968242304313516,
  'r2': 0.9844056210554766},
 'test': {'mae': 1572.4709275703044,
  'rmse': 2137.0592437238483,
  'smape': 0.015802987053363884,
  'r2': 0.9669471721281843}}

## Treinar modelos candidatos

Todos os candidatos são pipelines do `scikit-learn`.

A escolha privilegia modelos rápidos e explicáveis para aula.

Os modelos de árvore usam `n_jobs=1` de propósito.

No Windows, alguns ambientes `conda` podem emitir uma mensagem de `UnicodeDecodeError` em threads internas quando modelos usam subprocessos paralelos. A célula pode continuar funcionando, mas a mensagem assusta em aula. Usar `n_jobs=1` evita esse ruído e mantém a execução estável.

In [7]:
candidates = {
    "ridge_low_regularization": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=0.0001)),
        ]
    ),
    "hist_gradient_boosting": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, random_state=42)),
        ]
    ),
    "random_forest": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=1, min_samples_leaf=3)),
        ]
    ),
    "extra_trees": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(n_estimators=120, random_state=42, n_jobs=1, min_samples_leaf=3)),
        ]
    ),
}

trained_models = {}

for name, pipeline in candidates.items():
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    valid_pred = pipeline.predict(X_valid)
    test_pred = pipeline.predict(X_test)
    results.append(
        {
            "model": name,
            "valid": calculate_metrics(y_valid, valid_pred),
            "test": calculate_metrics(y_test, test_pred),
        }
    )

comparison_rows = []
for item in results:
    row = {"model": item["model"]}
    for split in ["valid", "test"]:
        for metric, value in item[split].items():
            row[f"{split}_{metric}"] = value
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows).sort_values("valid_rmse")
comparison

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "d:\_sys\conda\envs\aula_4_mlops_bitcoin\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "d:\_sys\conda\envs\aula_4_mlops_bitcoin\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "d:\_sys\conda\envs\aula_4_mlops_bitcoin\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc6 in position 79: invalid continuation byte


,model,valid_mae,valid_rmse,valid_smape,valid_r2,test_mae,test_rmse,test_smape,test_r2
0,naive_current_price,1316.743436,1834.794949,0.019968,0.984406,1572.470928,2137.059244,0.015803,0.966947
1,ridge_low_regularization,1332.834638,1855.114421,0.020131,0.984058,1636.548432,2195.196037,0.016403,0.965124
3,random_forest,6794.350245,12634.258932,0.092076,0.260578,37584.151354,39371.960831,0.445636,-10.218882
4,extra_trees,6893.494960,12827.753446,0.093434,0.237756,38381.965561,40114.119958,0.457522,-10.645819
2,hist_gradient_boosting,7623.713279,13529.986908,0.104973,0.152016,39562.557559,41254.295978,0.475277,-11.317253


In [8]:
non_baseline_comparison = comparison[comparison["model"] != "naive_current_price"].copy()
best_model_name = non_baseline_comparison.iloc[0]["model"]
best_model = trained_models[best_model_name]

best_valid_pred = best_model.predict(X_valid)
best_test_pred = best_model.predict(X_test)
best_metrics = {
    "selected_model": best_model_name,
    "validation": calculate_metrics(y_valid, best_valid_pred),
    "test": calculate_metrics(y_test, best_test_pred),
    "baseline_validation": calculate_metrics(y_valid, baseline_valid),
    "baseline_test": calculate_metrics(y_test, baseline_test),
}

print(json.dumps(best_metrics, indent=2, ensure_ascii=False))
if comparison.iloc[0]["model"] == "naive_current_price":
    print("O baseline foi melhor na validação. Mesmo assim, registramos o melhor modelo treinado para mostrar o ciclo MLOps completo.")


{
  "selected_model": "ridge_low_regularization",
  "validation": {
    "mae": 1332.8346383005755,
    "rmse": 1855.1144214331703,
    "smape": 0.020130507221293094,
    "r2": 0.9840583079756337
  },
  "test": {
    "mae": 1636.5484324560864,
    "rmse": 2195.196037425424,
    "smape": 0.016402902533703637,
    "r2": 0.9651243654359817
  },
  "baseline_validation": {
    "mae": 1316.7434360270856,
    "rmse": 1834.7949489867306,
    "smape": 0.019968242304313516,
    "r2": 0.9844056210554766
  },
  "baseline_test": {
    "mae": 1572.4709275703044,
    "rmse": 2137.0592437238483,
    "smape": 0.015802987053363884,
    "r2": 0.9669471721281843
  }
}
O baseline foi melhor na validação. Mesmo assim, registramos o melhor modelo treinado para mostrar o ciclo MLOps completo.


## Salvar artefatos locais

Esses arquivos ficam em `runtime/artifacts/`.

| Arquivo | Uso na aula |
|---|---|
| `feature_columns.json` | Lista de colunas que a API espera no payload. |
| `metrics_initial.json` | Métricas do treino inicial. |
| `sample_payload.json` | Exemplo de requisição para `/predict`. |
| `predictions_test.csv` | Previsões no conjunto de teste. |
| `model_metadata.json` | Metadados didáticos do modelo escolhido. |
| `local_model.joblib` | Cópia local do pipeline para inspeção e fallback didático. |


In [9]:
feature_path = ARTIFACTS_DIR / "feature_columns.json"
metrics_path = ARTIFACTS_DIR / "metrics_initial.json"
sample_path = ARTIFACTS_DIR / "sample_payload.json"
predictions_path = ARTIFACTS_DIR / "predictions_test.csv"
metadata_path = ARTIFACTS_DIR / "model_metadata.json"
local_model_path = ARTIFACTS_DIR / "local_model.joblib"

feature_path.write_text(json.dumps({"feature_columns": feature_columns}, indent=2, ensure_ascii=False), encoding="utf-8")
metrics_path.write_text(json.dumps(best_metrics, indent=2, ensure_ascii=False), encoding="utf-8")

sample_payload = {"features": X_test.iloc[-1].astype(float).to_dict()}
sample_path.write_text(json.dumps(sample_payload, indent=2, ensure_ascii=False), encoding="utf-8")

predictions = test_data[["time", "target_price_usd_d1"]].copy()
predictions["prediction"] = best_test_pred
predictions["baseline_prediction"] = baseline_test
predictions.to_csv(predictions_path, index=False)

joblib.dump(best_model, local_model_path)

metadata = {
    "model_name": MODEL_NAME,
    "selected_model": best_model_name,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "target": "target_price_usd_d1",
    "feature_columns_path": str(feature_path.relative_to(LESSON_DIR)),
    "local_model_path": str(local_model_path.relative_to(LESSON_DIR)),
}
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

print("Artefatos salvos em:", ARTIFACTS_DIR)

Artefatos salvos em: c:\Users\apcs1\Downloads\Operações de Aprendizado de Machine Learning\4_Ciclo_Vida_MLOps_Bitcoin - Copia\runtime\artifacts


## Registrar no MLflow

O MLflow guarda parâmetros, métricas, artefatos e o modelo registrado.

Depois do registro, o alias `homol` aponta para a versão criada nesta execução. A API usa esse alias para carregar o modelo sem precisar alterar código.

Alguns avisos do MLflow podem aparecer durante o registro. Para esta aula, avisos sobre registry local, artefatos ou APIs internas não significam falha se a célula terminar e mostrar o `Run ID`, o `Model URI` e a versão apontada por `homol`.


In [10]:
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    client.create_experiment(
        EXPERIMENT_NAME,
        artifact_location=MLFLOW_RUNS_DIR.resolve().as_uri(),
    )
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"treino_inicial_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
input_example = X_train.head(3)
signature = infer_signature(input_example, best_model.predict(input_example))
model_version = None

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_param("target", "target_price_usd_d1")
    mlflow.log_param("selected_model", best_model_name)
    mlflow.log_param("train_start", str(train_data["time"].min().date()))
    mlflow.log_param("train_end", str(train_data["time"].max().date()))
    mlflow.log_param("valid_start", str(valid_data["time"].min().date()))
    mlflow.log_param("valid_end", str(valid_data["time"].max().date()))
    mlflow.log_param("test_start", str(test_data["time"].min().date()))
    mlflow.log_param("test_end", str(test_data["time"].max().date()))
    mlflow.log_param("feature_count", len(feature_columns))

    for split_name, split_metrics in [
        ("valid", best_metrics["validation"]),
        ("test", best_metrics["test"]),
        ("baseline_valid", best_metrics["baseline_validation"]),
        ("baseline_test", best_metrics["baseline_test"]),
    ]:
        for metric_name, value in split_metrics.items():
            mlflow.log_metric(f"{split_name}_{metric_name}", value)

    for path in [feature_path, metrics_path, sample_path, predictions_path, metadata_path]:
        mlflow.log_artifact(str(path))

    model_info = mlflow.sklearn.log_model(
        best_model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
        signature=signature,
        input_example=input_example,
        metadata={"feature_columns": json.dumps(feature_columns)},
        await_registration_for=60,
    )

    run_id = run.info.run_id
    model_version = getattr(model_info, "registered_model_version", None)
    print("Run ID:", run_id)
    print("Model URI:", model_info.model_uri)

try:
    if model_version is None:
        versions = client.search_model_versions(f"name = '{MODEL_NAME}'")
        matching_versions = [version for version in versions if version.run_id == run_id]
        if matching_versions:
            model_version = max(matching_versions, key=lambda version: int(version.version)).version

    if model_version is not None:
        client.set_registered_model_alias(MODEL_NAME, "homol", str(model_version))
        print(f"Alias homol atualizado para versão {model_version}.")
    else:
        print("Modelo logado, mas a versão registrada desta execução não foi encontrada.")
except Exception as exc:
    print("Não foi possível aplicar alias no Model Registry.")
    print(exc)


2026/06/13 07:23:41 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/13 07:23:41 INFO mlflow.store.db.utils: Updating database tables
2026/06/13 07:23:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/13 07:23:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: 738769cd9bb64cd7a00d9a20fb33119c
Model URI: models:/m-71d95270a91b4a0bbc041088473d32a3
Alias homol atualizado para versão 1.


Successfully registered model 'bitcoin_price_forecaster'.
Created version '1' of model 'bitcoin_price_forecaster'.


## Próximo passo

Depois de validar o modelo em homologação, o alias `prod` pode apontar para a versão aprovada.

Isso troca o modelo servido pela API sem alterar o código da API.